
1. Load EHR data
2. Inspect dataset structure
3. Check patient identifiers
4. Find missing values
5. Detect duplicates
6. Validate dates
7. Check clinical value ranges

Real-world goal:
Before building dashboards, reports, or machine learning models, healthcare analysts must verify data quality.


## What this code does
Loads a healthcare dataset into a pandas DataFrame.

## Why we use it
Almost every healthcare analytics project starts by loading EHR, claims, or registry data.

## Real-world example
A hospital exports patient encounter data into a CSV file.


In [3]:
# Import our data science tools
import pandas as pd  # Pandas is our main tool for working with tables
import numpy as np   # Numpy helps with numbers and math

# Load the dataset from a web link into a "DataFrame" (a table)
url = "/content/DUMMY_DATA.csv"
ehr_df = pd.read_csv(url)

# Display the top 5 rows to see what our data looks like
ehr_df.head()

,encounter_id,Hospital Death,Height,Weight,Ethnicity,Age,Gender,Diagnosis code,Albumin,Arterial_partial_pressure_co2,...,Resprate_rate,Sodium,Oxygen_saturation,Temperature,WBC,Urineoutput,Systolic_blood_pressure_max,Systolic_blood_pressure_min,Diastolic_blood_pressure_min,Diastolic_blood_pressure_max
0,34937,1,NaN,NaN,Hispanic,NaN,M,NaN,NaN,34.0,...,1147.0,5.0,68.0,22.0,300.0,259.0,66.0,52.8,NaN,NaN
1,483670,0,NaN,NaN,Asian,53.0,NaN,NaN,NaN,29.0,...,1063.0,5.0,17.0,87.0,228.0,47.0,118.0,94.4,3.2,4.8
2,150725,1,175.14,NaN,African American,NaN,NaN,249.0,1.0,NaN,...,NaN,10.0,NaN,45.0,NaN,281.0,171.0,136.8,NaN,NaN
3,278547,0,183.08,NaN,African American,NaN,NaN,NaN,6.0,68.0,...,1085.0,12.0,14.0,NaN,NaN,86.0,177.0,141.6,4.0,6.0
4,457468,0,162.62,73.88,African American,58.0,NaN,NaN,6.0,225.0,...,NaN,3.0,14.0,79.0,NaN,15.0,158.0,126.4,0.8,1.2


## Pattern to Remember

```python
import pandas as pd
df = pd.read_csv("file.csv")
df.head()
```


# Step 1: Inspect Dataset Structure

## Why?
Learn number of rows, columns, and data types.

## Healthcare Example
Poor data quality can create wrong dashboards and unreliable models.

In [4]:
print(ehr_df.shape)
ehr_df.info()

(1000, 36)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 36 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   encounter_id                   1000 non-null   int64  
 1   Hospital Death                 1000 non-null   int64  
 2   Height                         339 non-null    float64
 3   Weight                         340 non-null    float64
 4   Ethnicity                      1000 non-null   object 
 5   Age                            330 non-null    float64
 6   Gender                         346 non-null    object 
 7   Diagnosis code                 488 non-null    float64
 8   Albumin                        700 non-null    float64
 9   Arterial_partial_pressure_co2  700 non-null    float64
 10  Arterial_ph                    700 non-null    float64
 11  Bilirubin                      700 non-null    float64
 12  BUN                            700 non

## Pattern to Remember

```python
print(ehr_df.shape)
ehr_df.info()
```


# Step 2: Evaluate Identifier Quality

## Why?
Check missing IDs and unique patients.

## Healthcare Example
Poor data quality can create wrong dashboards and unreliable models.

In [6]:
ehr_df[['encounter_id']].isnull().sum()
ehr_df['encounter_id'].nunique()

999

## Pattern to Remember

```python
ehr_df[['patient_id','encounter_id']].isnull().sum()
ehr_df['patient_id'].nunique()
```


# Step 3: Analyze Missing Data

## Why?
Find columns with missing values.

## Healthcare Example
Poor data quality can create wrong dashboards and unreliable models.

In [7]:
ehr_df.isnull().sum()

,0
encounter_id,0
Hospital Death,0
Height,661
Weight,660
Ethnicity,0
Age,670
Gender,654
Diagnosis code,512
Albumin,300
Arterial_partial_pressure_co2,300


## Pattern to Remember

```python
ehr_df.isnull().sum()
```


# Step 4: Detect Duplicate Records

## Why?
Find duplicated hospital records.

## Healthcare Example
Poor data quality can create wrong dashboards and unreliable models.

In [8]:
ehr_df.duplicated().sum()

np.int64(0)

## Pattern to Remember

```python
ehr_df.duplicated().sum()
```


# Step 5: Validate Temporal Consistency

## Why?
Find impossible dates.

## Healthcare Example
Poor data quality can create wrong dashboards and unreliable models.

In [12]:
# Add 'admission_time' column based on 'encounter_id'
np.random.seed(42) # for reproducibility
# Convert 'encounter_id' to a base datetime, then add random days
ehr_df['admission_time'] = pd.to_datetime('2023-01-01') + pd.to_timedelta(ehr_df['encounter_id'] % 365, unit='days')

# Introduce some invalid admission times (e.g., in the future or very old)
num_invalid_admissions = 50
invalid_admission_indices = np.random.choice(ehr_df.index, num_invalid_admissions, replace=False)
ehr_df.loc[invalid_admission_indices, 'admission_time'] = pd.to_datetime('2025-01-01') + pd.to_timedelta(ehr_df.loc[invalid_admission_indices, 'encounter_id'] % 365, unit='days')

print("Added 'admission_time' column.")
print(ehr_df[['encounter_id', 'admission_time']].head())

Added 'admission_time' column.
   encounter_id admission_time
0         34937     2023-09-20
1        483670     2023-02-15
2        150725     2023-12-12
3        278547     2023-02-22
4        457468     2023-05-04


In [13]:
# Add 'discharge_time' column based on 'admission_time' plus a random stay duration
np.random.seed(42) # for reproducibility
ehr_df['discharge_time'] = ehr_df['admission_time'] + pd.to_timedelta(np.random.randint(1, 30, size=len(ehr_df)), unit='days')

# Introduce some invalid discharge times (e.g., before admission)
num_invalid_discharges = 30
invalid_discharge_indices = np.random.choice(ehr_df.index, num_invalid_discharges, replace=False)
# Make discharge time earlier than admission time for these entries
ehr_df.loc[invalid_discharge_indices, 'discharge_time'] = ehr_df.loc[invalid_discharge_indices, 'admission_time'] - pd.to_timedelta(np.random.randint(1, 10, size=num_invalid_discharges), unit='days')

print("Added 'discharge_time' column.")
print(ehr_df[['admission_time', 'discharge_time']].head())

Added 'discharge_time' column.
  admission_time discharge_time
0     2023-09-20     2023-09-27
1     2023-02-15     2023-03-07
2     2023-12-12     2024-01-10
3     2023-02-22     2023-03-09
4     2023-05-04     2023-05-15


Now that the `admission_time` and `discharge_time` columns have been added, we can check for temporal inconsistencies.

In [14]:
ehr_df[ehr_df['discharge_time'] < ehr_df['admission_time']]

,encounter_id,Hospital Death,Height,Weight,Ethnicity,Age,Gender,Diagnosis code,Albumin,Arterial_partial_pressure_co2,...,Oxygen_saturation,Temperature,WBC,Urineoutput,Systolic_blood_pressure_max,Systolic_blood_pressure_min,Diastolic_blood_pressure_min,Diastolic_blood_pressure_max,admission_time,discharge_time
29,177357,0,NaN,NaN,Caucasian,NaN,NaN,208.0,NaN,15.0,...,57.0,60.0,180.0,NaN,NaN,NaN,0.0,0.0,2023-11-29,2023-11-23
70,193815,0,NaN,70.66,Caucasian,28.0,NaN,NaN,1.0,173.0,...,21.0,35.0,13.0,6.0,24.0,19.2,1.6,2.4,2023-01-01,2022-12-24
134,322362,1,NaN,NaN,African American,NaN,M,241.0,NaN,NaN,...,78.0,NaN,36.0,NaN,151.0,120.8,3.2,4.8,2023-03-09,2023-03-02
149,33764,1,NaN,75.06,Asian,77.0,NaN,405.0,3.0,15.0,...,NaN,65.0,153.0,NaN,91.0,72.8,2.4,3.6,2023-07-04,2023-06-27
205,500031,0,NaN,NaN,Hispanic,NaN,M,NaN,2.0,NaN,...,68.0,46.0,22.0,3.0,21.0,16.8,NaN,NaN,2023-12-13,2023-12-12
270,81216,1,203.85,NaN,African American,70.0,NaN,NaN,NaN,NaN,...,NaN,58.0,217.0,235.0,42.0,33.6,NaN,NaN,2023-07-06,2023-07-04
293,392186,1,NaN,NaN,Hispanic,NaN,NaN,NaN,NaN,207.0,...,3.0,85.0,288.0,232.0,95.0,76.0,1.6,2.4,2023-06-26,2023-06-25
344,147408,1,NaN,57.29,Hispanic,NaN,NaN,NaN,NaN,222.0,...,23.0,NaN,NaN,300.0,33.0,26.4,NaN,NaN,2023-11-10,2023-11-04
365,427233,1,NaN,70.72,Hispanic,18.0,NaN,NaN,NaN,177.0,...,7.0,91.0,NaN,NaN,176.0,140.8,1.6,2.4,2023-07-03,2023-06-25
374,128191,0,NaN,49.16,Asian,NaN,F,496.0,5.0,115.0,...,72.0,22.0,160.0,248.0,NaN,NaN,4.0,6.0,2023-03-18,2023-03-12


## Pattern to Remember

```python
ehr_df[ehr_df['discharge_time'] < ehr_df['admission_time']]
```


# Step 6: Inspect Clinical Value Ranges

## Why?
Check abnormal clinical values.

## Healthcare Example
Poor data quality can create wrong dashboards and unreliable models.

In [11]:
ehr_df[['heart_rate', 'Systolic_blood_pressure_max', 'Systolic_blood_pressure_min', 'Diastolic_blood_pressure_min', 'Diastolic_blood_pressure_max']].describe()

,heart_rate,Systolic_blood_pressure_max,Systolic_blood_pressure_min,Diastolic_blood_pressure_min,Diastolic_blood_pressure_max
count,700.000000,700.000000,700.000000,700.000000,700.000000
mean,810.030000,109.747143,87.797714,2.148571,3.222857
std,461.959452,60.097206,48.077765,1.262505,1.893757
min,6.000000,1.000000,0.800000,0.000000,0.000000
25%,428.500000,60.000000,48.000000,0.800000,1.200000
50%,794.500000,108.500000,86.800000,2.400000,3.600000
75%,1198.500000,162.000000,129.600000,3.200000,4.800000
max,1619.000000,214.000000,171.200000,4.000000,6.000000


## Pattern to Remember

```python
ehr_df[['heart_rate','systolic_bp','diastolic_bp']].describe()
```
